In [ ]:
# Resolve configurable_mdp root for common working directories
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    module_root = cwd
elif (cwd / "configurable_mdp" / "src").exists():
    module_root = cwd / "configurable_mdp"
elif cwd.name == "notebooks" and (cwd.parent / "src").exists():
    module_root = cwd.parent
else:
    raise RuntimeError(f"Could not locate configurable_mdp root from cwd={cwd}")

module_path = str(module_root)
if module_path not in sys.path:
    sys.path.insert(0, module_path)

print(f"module_path = {module_path}")
print(f"src import ready: {(Path(module_path) / 'src').exists()}")

In [ ]:
import os
import re
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import orbax

# JAX: force CPU for this notebook kernel session only
os.environ["JAX_PLATFORMS"] = "cpu"
import jax
jax.config.update("jax_platform_name", "cpu")

import jax.numpy as jnp
from src.algorithms.value_iteration_and_prediction import general_value_iteration
from train_four_rooms_hpgd import environment_setup
from visualization_functions import plot_UL_rewards, plot_incentive_grid

# Parameters for plotting
linewidth = 2.5
figsize = (8.0, 4.8)  # slightly larger graph box while keeping readability
base_font_size = 20
legend_font_size = 18  # small-caps labels look larger, so keep legend slightly smaller
axis_label_font_size = 22  # make axis labels slightly larger
solid_alpha = 0.9
shaded_alpha = 0.1
plt.rcParams.update(
    {
        'font.size': base_font_size,
        'legend.fontsize': legend_font_size,
        'axes.labelsize': axis_label_font_size,
        'text.usetex': True,
        'axes.linewidth': linewidth,
        'xtick.major.width': linewidth,
        'ytick.major.width': linewidth,
        'xtick.major.size': 2*linewidth,
        'ytick.major.size': 2*linewidth,
        'axes.prop_cycle': plt.cycler(color=plt.cm.Dark2.colors),
        "lines.linewidth": linewidth,
    }
)
# color_set = plt.cm.Dark2.colors
import numpy as np
color_set = plt.cm.viridis(np.linspace(0, 1, 9))  # Max 9 distinct colors. color_set[0]: vivid <-> color_set[-1]: faded
line_set = ['-', '--', '-.', (0, (3, 1, 1, 1)), ':', (0, (1, 3))]  # Max 6 distinct line styles

In [ ]:
# Reset-Q
data_dirs = [
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_50",
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_20",
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_10",
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_5",
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_2",
    "../data/subopt_reset_q/experiment_reg_lambda_0_003_total_steps_100_subopt_1",
]
# Carry-Q
# data_dirs = [
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_50",
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_20",
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_10",
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_5",
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_2",
#     "../data/subopt_carry_q/experiment_reg_lambda_0_003_total_steps_100_subopt_1",
# ]

save_figs = True
save_dir = os.getcwd()

algorithms = ["bchg", "hpgd", "hpgd_sarsa", "sobirl", "baseline", "hpgd_oracle"]

main_metric = "UL_initial_value_opt"

# Other metrics
other_metrics_and_labels = {
    "kl_optimality_gap": "Lower-level KL optimality gap",
}
y_scale = {
    "kl_optimality_gap": ["linear", "log"],
}

# Extract suboptimality labels and create mapping to directories
subopt_dirs = {}
for dataset_dir in data_dirs:
    match = re.search(r"subopt_\d+(?:_\d+)*$", os.path.basename(dataset_dir))
    subopt_label = match.group(0) if match else os.path.basename(dataset_dir)
    subopt_dirs[subopt_label] = dataset_dir
subopt_labels = list(subopt_dirs.keys())
subopt_order = list(subopt_dirs.keys())

# Use the first dataset as the common configuration anchor.
base_dataset_dir = subopt_dirs[subopt_labels[0]]
match = re.search(r"experiment_reg_lambda_(.+?)subopt", os.path.basename(base_dataset_dir))
fig_label = "_beta_" + match.group(1) + "subopt" if match else "_beta"  # Common label for figure filenames

# Plot settings
max_outer_iterations = 10_000
rolling_window = 100
plot_top = 10
seed_idx = None
subopt_legend_names = {
    subopt_label: subopt_label.split("subopt_", 1)[1] if "subopt_" in subopt_label else subopt_label
    for subopt_label in subopt_labels
}
line_styles = {
    subopt_label: line_set[i % len(line_set)]
    for i, subopt_label in enumerate(subopt_labels)
}
algo_colors = {
    subopt_label: color_set[i % len(color_set)]
    for i, subopt_label in enumerate(subopt_labels)
}
# Note: the order in the legend follows the order in list "subopt_order"
zorder = dict(zip(subopt_order, [len(subopt_order) - i for i in range(len(subopt_order))]))
algo_labels = {
    "bchg": r"\textsc{BC-HG}",
    "hpgd": r"\textsc{HPGD (MC)}",
    "hpgd_sarsa": r"\textsc{HPGD (SA)}",
    "hpgd_oracle": r"\textsc{HPGD (oracle)}",
    "sobirl": r"\textsc{SoBiRL}",
    "baseline": r"\textsc{Naive-PGD}",
}

# Load the data

In [ ]:
# --- Loading Data --- #

grid_search_outputs = {algo: {} for algo in algorithms}
grid_search_params = {algo: {} for algo in algorithms}
grid_search_train_states = {algo: {} for algo in algorithms}
summary_dfs = {algo: {} for algo in algorithms}
best_parameters_for_reg_lambda = {algo: {} for algo in algorithms}

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

with open(f"{base_dataset_dir}/config.yaml", "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)
print(config)
rng = jax.random.PRNGKey(config["random_seed"])

# Environment Setup
print("Environment Setup")
env, env_params, incentive_train_state, config_incentive = environment_setup(rng, config)

for subopt_label, dataset_dir in subopt_dirs.items():
    print(f"\n=== Loading {subopt_label} ===")
    for algo in algorithms:
        try:
            print(f"--- {algo} ---")
            metrics_path = f"{dataset_dir}/metrics_{algo}.pkl" if algo != "bilevel" else f"{dataset_dir}/metrics_on_policy.pkl"
            with open(metrics_path, "rb") as f:
                outputs = pickle.load(f)
            grid_search_outputs[algo][subopt_label] = outputs
            with open(f"{dataset_dir}/grid_search_{algo}.pkl" if algo not in ["bilevel", "on_policy", "uniform"] else f"{dataset_dir}/grid_search.pkl", "rb") as f:
                grid_params = pd.DataFrame(pickle.load(f))
            grid_search_params[algo][subopt_label] = grid_params
            print("Output shape: ", outputs[main_metric].shape)
            print("Grid param keys: ", grid_params.columns)
            for k in outputs:
                if jnp.any(jnp.isnan(outputs[k])):
                    print(f"\033[91m[WARNING] Found NaN values in {k} for {algo} ({subopt_label})\033[0m")

            ckpt_path = f"{dataset_dir}/checkpoint_incentive_{algo}" if algo != "bilevel" else f"{dataset_dir}/checkpoint_incentive_on_policy"
            orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
            grid_search_train_states[algo][subopt_label] = orbax_checkpointer.restore(ckpt_path)
            print("Incentive Train State Loaded")
        except Exception as exc:
            print(f"Failed to load {algo} for {subopt_label}: {exc}")
            continue

        df = grid_params.copy()
        df[main_metric] = jnp.mean(outputs[main_metric][:, -1000:], axis=1)  # average across the last 1000 steps
        df = df.set_index(list(grid_params.keys()))
        df = df.groupby(list(grid_params.keys())).mean()  # average across seeds
        df = df.sort_values(main_metric, ascending=False)
        display(df)
        summary_dfs[algo][subopt_label] = df

In [ ]:
# Merge results
combined_results = pd.DataFrame()

for algo in algorithms:
    if algo not in summary_dfs:
        continue
    for subopt_key, df in summary_dfs[algo].items():
        df = df.copy().reset_index()
        df['algorithm'] = algo
        df['subopt'] = subopt_key
        combined_results = pd.concat([combined_results, df], ignore_index=True)

# Display a pivot table for each algorithm, with subopt levels as columns
if not combined_results.empty:
    pivot_table = combined_results.pivot_table(
        values=main_metric,
        index=['algorithm', 'incentive_reg_grid', 'lr_grid'],
        columns='subopt',
        aggfunc='first'
    )

    print("Combined Results Table (Algorithm x Subopt)")
    print("=" * 60)
    print(pivot_table.to_string())


# Performance Table

In [ ]:
# Find the best parameters for each reg_lambda
for algo in algorithms:
    available_subopts = [subopt_key for subopt_key in subopt_order if subopt_key in grid_search_params.get(algo, {})]
    if not available_subopts:
        print(f"Skipping {algo} because no subopt datasets were loaded")
        continue

    print(f"\n=== Performance Table: {algo} ===")
    print(f"params: {summary_dfs[algo][available_subopts[0]].index.names}")
    best_parameters_for_reg_lambda[algo] = {}

    reference_subopt = available_subopts[0]
    reg_lambdas = jnp.unique(grid_search_params[algo][reference_subopt]["incentive_reg_grid"].values)
    df_summary_mean = pd.DataFrame(index=[float(reg_lambda) for reg_lambda in reg_lambdas])
    df_summary_std_error = pd.DataFrame(index=[float(reg_lambda) for reg_lambda in reg_lambdas])

    for subopt_key in available_subopts:
        df_grid_params = grid_search_params[algo][subopt_key]
        df = summary_dfs[algo][subopt_key]
        outputs = grid_search_outputs[algo][subopt_key]
        best_params = {}

        for reg_lambda in reg_lambdas:
            reg_lambda = float(reg_lambda)
            df_selected = df.loc[(slice(None), reg_lambda), :]
            if df_selected.empty:
                print(f"Skipping {algo} / {subopt_key} at reg_lambda {reg_lambda} because no rows matched")
                continue
            best_params[reg_lambda] = df_selected.index[0]
            print(f"Best parameters for {algo} / {subopt_key} at reg_lambda {reg_lambda}: {best_params[reg_lambda]}")

            idx = (df_grid_params == best_params[reg_lambda]).all(1).values
            UL_estimate = jnp.mean(outputs[main_metric][idx, -1000:], -1)
            df_summary_mean.loc[reg_lambda, subopt_key] = jnp.mean(UL_estimate)
            df_summary_std_error.loc[reg_lambda, subopt_key] = jnp.std(UL_estimate) / jnp.sqrt(UL_estimate.shape[0])

        best_parameters_for_reg_lambda[algo][subopt_key] = best_params

    print("Average Upper-Level Value (Last 1000 steps)")
    display(df_summary_mean.sort_index())
    print("Std. Error Upper-Level Value (Last 1000 steps)")
    display(df_summary_std_error.sort_index())
    print("Note: The row index is incentive_reg_grid, the column index is subopt setting.")

# Visualize the learning curves

In [ ]:
for algo in algorithms:
    available_subopts = [subopt_key for subopt_key in subopt_order if subopt_key in grid_search_params.get(algo, {})]
    if not available_subopts:
        print(f"Skipping {algo} because no subopt datasets were loaded")
        continue

    print(f"\n=== Learning Curves: {algo} ===")
    reference_subopt = available_subopts[0]
    reg_lambdas = jnp.unique(grid_search_params[algo][reference_subopt]["incentive_reg_grid"].values)

    for reg_lambda in reg_lambdas:
        reg_lambda = float(reg_lambda)
        print(f"\n--- Reg_lambda: {reg_lambda} ---")
        input_data = {}
        for subopt_key in available_subopts:
            df_grid_params = grid_search_params[algo][subopt_key]
            outputs = grid_search_outputs[algo][subopt_key]
            idx = (df_grid_params == best_parameters_for_reg_lambda[algo][subopt_key][reg_lambda]).all(1).values
            input_data[subopt_key] = outputs[main_metric][idx] if seed_idx is None else outputs[main_metric][idx][seed_idx].reshape(1, -1)

        if float(reg_lambda).is_integer():
            filename = f"{main_metric}_{algo}_grid_search_reg_lambda_{int(reg_lambda)}{fig_label}.pdf"
        else:
            filename = f"{main_metric}_{algo}_grid_search_reg_lambda_{str(reg_lambda).replace('.', '_')}{fig_label}.pdf"

        plot_UL_rewards(
            input_data,
            figsize=figsize,
            rolling_window=rolling_window,
            title=algo_labels[algo],
            savefig_path=os.path.join(save_dir, filename) if save_figs else None,
            xlim=(0, max_outer_iterations),
            ylim=(-1.2, 1.6),
            legend_position={
                'loc': 'center right',
                'bbox_to_anchor': (0.975, 0.5),
                'ncol': 1
            },
            legend_names=subopt_legend_names,
            line_styles=line_styles,
            algo_colors=algo_colors,
            zorder=zorder,
            yscale="linear",
            distinct_fill_between_edge=False,
            alpha=solid_alpha,
            fill_alpha=shaded_alpha,
        )
        plt.close()
plt.clf()

# Visualize the Other Metrics

In [ ]:
for metric, y_label in other_metrics_and_labels.items():
    for y_scale_option in y_scale[metric]:

        if metric == "kl_optimality_gap":
            if y_scale_option == "linear":
                y_lim = (-0.1, 3.1)
            elif y_scale_option == "log":
                y_lim = (1e-11, 1e1)
        else:
            y_lim = None

        for algo in algorithms:
            available_subopts = [subopt_key for subopt_key in subopt_order if subopt_key in grid_search_params.get(algo, {})]
            if not available_subopts:
                continue

            print(f"\n=== Metric: {metric} | Algorithm: {algo} ===")
            reference_subopt = available_subopts[0]
            reg_lambdas = jnp.unique(grid_search_params[algo][reference_subopt]["incentive_reg_grid"].values)

            for reg_lambda in reg_lambdas:
                reg_lambda = float(reg_lambda)
                print(f"\n--- Reg_lambda: {reg_lambda} ---")
                input_data = {}
                for subopt_key in available_subopts:
                    df_grid_params = grid_search_params[algo][subopt_key]
                    outputs = grid_search_outputs[algo][subopt_key]
                    idx = (df_grid_params == best_parameters_for_reg_lambda[algo][subopt_key][reg_lambda]).all(1).values
                    if metric in outputs:
                        input_data[subopt_key] = outputs[metric][idx] if seed_idx is None else outputs[metric][idx][seed_idx].reshape(1, -1)
                    else:
                        print(f"Metric {metric} not found for {algo} / {subopt_key}, skipping.")
                        continue

                if float(reg_lambda).is_integer():
                    filename = f"{metric}_{algo}_grid_search_reg_lambda_{int(reg_lambda)}{fig_label}.pdf"
                else:
                    filename = f"{metric}_{algo}_grid_search_reg_lambda_{str(reg_lambda).replace('.', '_')}{fig_label}.pdf"
                if y_scale_option == "log":
                    filename = filename.replace(".pdf", "_logscale.pdf")

                plot_UL_rewards(
                    input_data,
                    figsize=figsize,
                    rolling_window=rolling_window,
                    title=algo_labels[algo],
                    savefig_path=os.path.join(save_dir, filename) if save_figs else None,
                    xlim=(0, max_outer_iterations),
                    ylim=y_lim if y_lim is not None else None,
                    y_ticks=None,
                    ylabel=y_label,
                    yscale=y_scale_option,
                    legend_position={
                        'loc': 'center right',
                        'bbox_to_anchor': (0.975, 0.5),
                        'ncol': 1
                    },
                    legend_names=subopt_legend_names,
                    line_styles=line_styles,
                    algo_colors=algo_colors,
                    zorder=zorder,
                    distinct_fill_between_edge=False,
                    alpha=solid_alpha,
                    fill_alpha=shaded_alpha,
                )
                plt.close()
        plt.clf()